# 📊 TelecomX LATAM – Análisis de Evasión de Clientes (Churn)

**Desafío:** One Oracle Next Education (ONE) – Alura Latam  
**Objetivo:** Analizar los factores que explican la evasión de clientes en una empresa de telecomunicaciones de LATAM e identificar estrategias para reducirla.  
**Dataset:** `TelecomX_Data.json` – 7,267 registros con información demográfica, de servicios contratados y de facturación.

---
## Estructura del Análisis
1. **Extracción** – Carga y aplanamiento del JSON anidado
2. **Transformación** – Limpieza, estandarización e ingeniería de características
3. **Carga y Análisis** – EDA, visualizaciones y matriz de correlación
4. **Informe Final** – Hallazgos, conclusiones y recomendaciones estratégicas

# 📌 Extracción

In [ ]:
# ============================================================
# SECCIÓN 1: EXTRACCIÓN
# Carga del JSON anidado y normalización a DataFrame tabular
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

# Configuración global de visualizaciones
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# 1.1 Carga del archivo JSON
# -----------------------------------------------------------
ruta_json = 'TelecomX_Data.json'

with open(ruta_json, 'r', encoding='utf-8') as archivo:
    datos_crudos = json.load(archivo)

print(f'✅ JSON cargado exitosamente.')
print(f'   Tipo de dato raíz : {type(datos_crudos)}')
print(f'   Número de registros: {len(datos_crudos):,}')
print()
print('Ejemplo de un registro (estructura anidada):')
print(json.dumps(datos_crudos[0], indent=2, ensure_ascii=False))

In [ ]:
# -----------------------------------------------------------
# 1.2 Normalización: aplanar el JSON anidado a tabla plana
#
# El JSON tiene sub-objetos: customer, phone, internet, account
# y account.Charges (Monthly, Total). Usamos pd.json_normalize
# con sep='.' para conservar los nombres jerárquicos.
# -----------------------------------------------------------
df_raw = pd.json_normalize(
    datos_crudos,
    sep='.',          # separa niveles con punto: account.Contract
    errors='ignore'   # ignora claves faltantes en registros parciales
)

print(f'✅ Normalización completada.')
print(f'   Filas   : {df_raw.shape[0]:,}')
print(f'   Columnas: {df_raw.shape[1]}')
print()
print('Columnas generadas tras la normalización:')
print(df_raw.columns.tolist())
print()
df_raw.head(3)

# 🔧 Transformación

In [ ]:
# ============================================================
# SECCIÓN 2: TRANSFORMACIÓN
# Renombrado, limpieza, tipado e ingeniería de características
# ============================================================

# Trabajamos sobre una copia para preservar los datos crudos
df = df_raw.copy()

# -----------------------------------------------------------
# 2.1 Renombrado de columnas: inglés → español
# Basado en el diccionario de datos TelecomX_diccionario.md
# -----------------------------------------------------------
mapa_columnas = {
    'customerID'                 : 'ID_Cliente',
    'Churn'                      : 'Evasion',
    'customer.gender'            : 'Genero',
    'customer.SeniorCitizen'     : 'Ciudadano_Mayor',
    'customer.Partner'           : 'Pareja',
    'customer.Dependents'        : 'Dependientes',
    'customer.tenure'            : 'Antiguedad',
    'phone.PhoneService'         : 'Servicio_Telefonico',
    'phone.MultipleLines'        : 'Multiples_Lineas',
    'internet.InternetService'   : 'Servicio_Internet',
    'internet.OnlineSecurity'    : 'Seguridad_Linea',
    'internet.OnlineBackup'      : 'Respaldo_Linea',
    'internet.DeviceProtection'  : 'Proteccion_Dispositivo',
    'internet.TechSupport'       : 'Soporte_Tecnico',
    'internet.StreamingTV'       : 'Streaming_TV',
    'internet.StreamingMovies'   : 'Streaming_Peliculas',
    'account.Contract'           : 'Tipo_Contrato',
    'account.PaperlessBilling'   : 'Factura_Digital',
    'account.PaymentMethod'      : 'Metodo_Pago',
    'account.Charges.Monthly'    : 'Cargos_Mensuales',
    'account.Charges.Total'      : 'Cargos_Totales',
}

df = df.rename(columns=mapa_columnas)

print('✅ Columnas renombradas al español:')
print(df.columns.tolist())

In [ ]:
# -----------------------------------------------------------
# 2.2 Limpieza crítica: Cargos_Totales
#
# Problema detectado: clientes con Antiguedad=0 tienen
# Cargos_Totales como string de espacio " " (vacío).
# Solución:
#   - Reemplazar " " → "0" cuando Antiguedad == 0
#   - Convertir toda la columna a float con pd.to_numeric
# -----------------------------------------------------------

# Identificar cuántos registros tienen el espacio problemático
espacios = df[df['Cargos_Totales'].astype(str).str.strip() == ''].shape[0]
print(f'⚠️  Registros con Cargos_Totales vacío (" "): {espacios}')

# Reemplazar espacios vacíos por "0" en clientes con Antiguedad=0
mask_nuevos = (df['Antiguedad'] == 0) & (df['Cargos_Totales'].astype(str).str.strip() == '')
df.loc[mask_nuevos, 'Cargos_Totales'] = '0'

# Convertir la columna a tipo numérico (float)
# Los valores no convertibles se asignan como NaN
df['Cargos_Totales'] = pd.to_numeric(df['Cargos_Totales'], errors='coerce')

# Verificar resultado
print(f'   Tipo final de Cargos_Totales: {df["Cargos_Totales"].dtype}')
print(f'   Valores NaN restantes        : {df["Cargos_Totales"].isnull().sum()}')
print(f'   Rango: ${df["Cargos_Totales"].min():.2f} – ${df["Cargos_Totales"].max():.2f}')

In [ ]:
# -----------------------------------------------------------
# 2.3 Conversiones de tipo y estandarización de valores
# -----------------------------------------------------------

# Ciudadano_Mayor: de entero (0/1) a texto legible (No/Sí)
df['Ciudadano_Mayor'] = df['Ciudadano_Mayor'].map({0: 'No', 1: 'Sí'})

# Evasion: estandarizar a español (Yes→Sí, No→No)
df['Evasion'] = df['Evasion'].map({'Yes': 'Sí', 'No': 'No'})

# Asegurar que Cargos_Mensuales sea numérico
df['Cargos_Mensuales'] = pd.to_numeric(df['Cargos_Mensuales'], errors='coerce')

# Asegurar que Antiguedad sea entero
df['Antiguedad'] = pd.to_numeric(df['Antiguedad'], errors='coerce').astype('Int64')

print('✅ Tipos de dato actualizados:')
print(df[['ID_Cliente','Evasion','Ciudadano_Mayor','Antiguedad',
          'Cargos_Mensuales','Cargos_Totales']].dtypes)

In [ ]:
# -----------------------------------------------------------
# 2.4 Ingeniería de Características
# -----------------------------------------------------------

# Feature 1: Cuentas_Diarias
# Costo promedio diario = cargo mensual / 30 días
df['Cuentas_Diarias'] = (df['Cargos_Mensuales'] / 30).round(2)

# Feature 2: Total_Servicios
# Suma de los servicios opcionales activos ('Yes') por cliente
# Incluye: Servicio_Telefonico, Multiples_Lineas, Seguridad_Linea,
#          Respaldo_Linea, Proteccion_Dispositivo, Soporte_Tecnico,
#          Streaming_TV, Streaming_Peliculas
columnas_servicios = [
    'Servicio_Telefonico',
    'Multiples_Lineas',
    'Seguridad_Linea',
    'Respaldo_Linea',
    'Proteccion_Dispositivo',
    'Soporte_Tecnico',
    'Streaming_TV',
    'Streaming_Peliculas',
]

df['Total_Servicios'] = df[columnas_servicios].apply(
    lambda fila: (fila == 'Yes').sum(), axis=1
)

print('✅ Características creadas:')
print(f'   Cuentas_Diarias  → ${df["Cuentas_Diarias"].mean():.2f} promedio/día')
print(f'   Total_Servicios  → {df["Total_Servicios"].mean():.2f} servicios en promedio por cliente')
print()
print('Distribución de Total_Servicios:')
print(df['Total_Servicios'].value_counts().sort_index())

In [ ]:
# -----------------------------------------------------------
# 2.5 Verificación de calidad de datos
# -----------------------------------------------------------

print('=' * 55)
print('  REPORTE DE CALIDAD DE DATOS')
print('=' * 55)
print(f'  Filas        : {df.shape[0]:,}')
print(f'  Columnas     : {df.shape[1]}')
print(f'  Duplicados   : {df.duplicated().sum()}')
print()

# Valores nulos por columna
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]
if nulos.empty:
    print('  Valores nulos: ninguno ✅')
else:
    print('  Columnas con nulos:')
    print(nulos)

print()
print('Tipos de dato finales:')
print(df.dtypes)
print()
print('Vista previa del DataFrame transformado:')
df.head(5)

# 📊 Carga y análisis

In [ ]:
# ============================================================
# SECCIÓN 3: CARGA Y ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ============================================================

# -----------------------------------------------------------
# 3.1 Distribución de la variable objetivo: Evasión (Churn)
# -----------------------------------------------------------
conteo_evasion = df['Evasion'].value_counts()
pct_evasion    = df['Evasion'].value_counts(normalize=True) * 100

resumen = pd.DataFrame({
    'Clientes' : conteo_evasion,
    'Porcentaje': pct_evasion.round(2)
})

print('=' * 40)
print('  DISTRIBUCIÓN DE EVASIÓN DE CLIENTES')
print('=' * 40)
print(resumen.to_string())
print()

tasa = pct_evasion.get('Sí', pct_evasion.get('Yes', 0))
print(f'📌 Tasa global de evasión (Churn): {tasa:.1f}%')

In [ ]:
# -----------------------------------------------------------
# 3.2 Estadísticas descriptivas de variables numéricas
# -----------------------------------------------------------
columnas_num = ['Antiguedad', 'Cargos_Mensuales', 'Cargos_Totales',
                'Cuentas_Diarias', 'Total_Servicios']

print('Estadísticas descriptivas (variables numéricas):')
df[columnas_num].describe().round(2)

In [ ]:
# -----------------------------------------------------------
# 3.3 Matriz de Correlación – Heatmap
# Identifica drivers numéricos de Evasión
# Nota: Evasión se codifica temporalmente como 1=Sí, 0=No
# -----------------------------------------------------------

df_corr = df[columnas_num].copy()
df_corr['Evasion_num'] = (df['Evasion'] == 'Sí').astype(int)

matriz_corr = df_corr.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))

sns.heatmap(
    matriz_corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    linecolor='white',
    square=True,
    cbar_kws={'shrink': 0.8},
    ax=ax
)

ax.set_title(
    'Matriz de Correlación\nDrivers de Evasión de Clientes',
    fontsize=14, fontweight='bold', pad=15
)
plt.tight_layout()
plt.savefig('correlacion_evasion.png', dpi=150, bbox_inches='tight')
plt.show()

# Extraer correlaciones con la variable Evasion_num
print('\nCorrelaciones con Evasión (ordenadas):')
print(matriz_corr['Evasion_num'].drop('Evasion_num').sort_values(ascending=False))

In [ ]:
# -----------------------------------------------------------
# 3.4 Visualización 1: Evasión vs Tipo de Contrato
# Hipótesis: los contratos mes a mes tienen mayor evasión
# -----------------------------------------------------------

# Calcular porcentaje de evasión por tipo de contrato
contrato_evasion = (
    df.groupby(['Tipo_Contrato', 'Evasion'])
      .size()
      .unstack(fill_value=0)
)
contrato_pct = contrato_evasion.div(contrato_evasion.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico izquierdo: conteo absoluto
contrato_evasion.plot(
    kind='bar',
    ax=axes[0],
    color=['#4C9BE8', '#E8784C'],
    edgecolor='white',
    width=0.6
)
axes[0].set_title('Evasión por Tipo de Contrato\n(Volumen absoluto)', fontweight='bold')
axes[0].set_xlabel('Tipo de Contrato')
axes[0].set_ylabel('Número de Clientes')
axes[0].legend(title='Evasión')
axes[0].tick_params(axis='x', rotation=15)

# Gráfico derecho: porcentaje de evasión
if 'Sí' in contrato_pct.columns:
    col_si = 'Sí'
else:
    col_si = 'Yes'

bars = axes[1].bar(
    contrato_pct.index,
    contrato_pct[col_si],
    color=['#E8784C', '#4C9BE8', '#5DBE8A'],
    edgecolor='white',
    width=0.5
)

# Etiquetas sobre las barras
for bar, val in zip(bars, contrato_pct[col_si]):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.1f}%',
        ha='center', va='bottom', fontweight='bold', fontsize=12
    )

axes[1].set_title('Tasa de Evasión (%) por Tipo de Contrato', fontweight='bold')
axes[1].set_xlabel('Tipo de Contrato')
axes[1].set_ylabel('Tasa de Evasión (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis='x', rotation=15)
axes[1].set_ylim(0, 55)

plt.suptitle('Análisis de Evasión por Tipo de Contrato', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evasion_contrato.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -----------------------------------------------------------
# 3.5 Visualización 2: Evasión vs Antigüedad del cliente
# Hipótesis: los clientes más nuevos evaden más
# -----------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma superpuesto por grupo de evasión
for grupo, color in zip(['No', 'Sí'], ['#4C9BE8', '#E8784C']):
    subset = df[df['Evasion'] == grupo]['Antiguedad'].dropna()
    axes[0].hist(
        subset,
        bins=35,
        alpha=0.65,
        color=color,
        label=f'Evasión = {grupo}',
        edgecolor='white'
    )

axes[0].set_title('Distribución de Antigüedad\npor Grupo de Evasión', fontweight='bold')
axes[0].set_xlabel('Antigüedad (meses)')
axes[0].set_ylabel('Número de Clientes')
axes[0].legend()

# KDE plot (densidad de probabilidad)
for grupo, color in zip(['No', 'Sí'], ['#4C9BE8', '#E8784C']):
    subset = df[df['Evasion'] == grupo]['Antiguedad'].dropna().astype(float)
    sns.kdeplot(
        data=subset,
        ax=axes[1],
        color=color,
        fill=True,
        alpha=0.35,
        linewidth=2,
        label=f'Evasión = {grupo}'
    )

axes[1].set_title('Densidad de Antigüedad\npor Grupo de Evasión (KDE)', fontweight='bold')
axes[1].set_xlabel('Antigüedad (meses)')
axes[1].set_ylabel('Densidad')
axes[1].legend()

plt.suptitle('Relación entre Antigüedad y Evasión de Clientes', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evasion_antiguedad.png', dpi=150, bbox_inches='tight')
plt.show()

# Estadísticas rápidas
print('Antigüedad promedio por grupo:')
print(df.groupby('Evasion')['Antiguedad'].mean().round(1))

In [ ]:
# -----------------------------------------------------------
# 3.6 Visualización 3: Evasión vs Cargos Mensuales
# Hipótesis: los clientes con mayor cargo mensual evaden más
# -----------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot: distribución de cargos mensuales por grupo de evasión
sns.boxplot(
    data=df,
    x='Evasion',
    y='Cargos_Mensuales',
    palette=['#4C9BE8', '#E8784C'],
    order=['No', 'Sí'],
    width=0.5,
    linewidth=1.5,
    ax=axes[0]
)
axes[0].set_title('Distribución de Cargos Mensuales\npor Grupo de Evasión', fontweight='bold')
axes[0].set_xlabel('Evasión')
axes[0].set_ylabel('Cargos Mensuales (USD)')
axes[0].yaxis.set_major_formatter(mtick.FormatStrFormatter('$%.0f'))

# Violinplot: distribución completa
sns.violinplot(
    data=df,
    x='Evasion',
    y='Cargos_Mensuales',
    palette=['#4C9BE8', '#E8784C'],
    order=['No', 'Sí'],
    inner='quartile',
    linewidth=1.5,
    ax=axes[1]
)
axes[1].set_title('Distribución de Cargos Mensuales\n(Violin Plot)', fontweight='bold')
axes[1].set_xlabel('Evasión')
axes[1].set_ylabel('Cargos Mensuales (USD)')
axes[1].yaxis.set_major_formatter(mtick.FormatStrFormatter('$%.0f'))

plt.suptitle('Relación entre Cargos Mensuales y Evasión de Clientes',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evasion_cargos.png', dpi=150, bbox_inches='tight')
plt.show()

# Estadísticas de respaldo
print('Cargos mensuales promedio por grupo:')
print(df.groupby('Evasion')['Cargos_Mensuales'].agg(['mean', 'median']).round(2))

In [ ]:
# -----------------------------------------------------------
# 3.7 Análisis complementario: Total_Servicios vs Evasión
# -----------------------------------------------------------

servicios_evasion = (
    df.groupby(['Total_Servicios', 'Evasion'])
      .size()
      .unstack(fill_value=0)
      .assign(Tasa_Evasion=lambda x: x.get('Sí', x.get('Yes', 0)) / x.sum(axis=1) * 100)
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    servicios_evasion.index,
    servicios_evasion['Tasa_Evasion'],
    color=sns.color_palette('Blues_d', len(servicios_evasion)),
    edgecolor='white'
)

ax.set_title('Tasa de Evasión según Número Total de Servicios Contratados',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Total de Servicios Contratados')
ax.set_ylabel('Tasa de Evasión (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# Añadir etiquetas
for i, val in enumerate(servicios_evasion['Tasa_Evasion']):
    ax.text(i, val + 0.3, f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('evasion_servicios.png', dpi=150, bbox_inches='tight')
plt.show()

# 📄 Informe final

## 1. Introducción

### Contexto del Problema

TelecomX es una empresa de telecomunicaciones que opera en América Latina y enfrenta un problema crítico de negocios: la **evasión de clientes** (también conocida en la industria como *customer churn*). Perder un cliente implica no solo la pérdida inmediata de ingresos recurrentes, sino también costos de adquisición significativos para reemplazarlo. En el sector de telecomunicaciones, el costo de retener un cliente existente puede ser **cinco veces menor** al de capturar uno nuevo.

### Objetivo del Análisis

Este análisis tiene como propósito:
1. **Identificar los factores** demográficos, de servicios y de facturación que más se asocian con la evasión.
2. **Cuantificar la magnitud** del problema actual.
3. **Proponer recomendaciones estratégicas** accionables para los equipos de producto, marketing y retención.

### Dataset
- **Fuente:** `TelecomX_Data.json`
- **Registros analizados:** 7,267 clientes
- **Variables:** 21 atributos originales + 2 features creadas (`Cuentas_Diarias`, `Total_Servicios`)
- **Período:** Datos de corte transversal (snapshot) de la base de clientes activos e inactivos.

In [ ]:
# -----------------------------------------------------------
# Cálculos de apoyo para el Informe Ejecutivo
# (Se ejecutan aquí para tener los números en el Markdown)
# -----------------------------------------------------------
tasa_churn   = (df['Evasion'] == 'Sí').mean() * 100
ant_si       = df[df['Evasion'] == 'Sí']['Antiguedad'].mean()
ant_no       = df[df['Evasion'] == 'No']['Antiguedad'].mean()
cargo_si     = df[df['Evasion'] == 'Sí']['Cargos_Mensuales'].mean()
cargo_no     = df[df['Evasion'] == 'No']['Cargos_Mensuales'].mean()

# Tasa de evasión por tipo de contrato
tasa_por_contrato = (
    df.groupby('Tipo_Contrato')
      .apply(lambda g: (g['Evasion'] == 'Sí').mean() * 100)
      .round(1)
      .sort_values(ascending=False)
)

print('📊 KPIs CLAVE DEL ANÁLISIS')
print('=' * 45)
print(f'  Tasa global de evasión      : {tasa_churn:.1f}%')
print(f'  Antigüedad media (evaden)   : {ant_si:.1f} meses')
print(f'  Antigüedad media (retienen) : {ant_no:.1f} meses')
print(f'  Cargo mensual medio (evaden): ${cargo_si:.2f}')
print(f'  Cargo mensual medio (retienen): ${cargo_no:.2f}')
print()
print('Tasa de evasión por tipo de contrato:')
print(tasa_por_contrato.to_string())

## 2. Análisis de Hallazgos

### 2.1 Magnitud de la Evasión

Aproximadamente **el 26% de los clientes del dataset abandonaron la empresa** (tasa de churn). Esta cifra está por encima del promedio histórico de la industria de telecomunicaciones en LATAM (~20%), lo que indica que el problema es urgente y requiere atención inmediata.

---

### 2.2 Hallazgo 1 – El Tipo de Contrato es el Principal Predictor

La variable con mayor impacto en la evasión es el **tipo de contrato**:

| Tipo de Contrato | Tasa de Evasión |
|---|---|
| Month-to-month (mensual) | ~**43%** |
| One year (anual) | ~**11%** |
| Two year (bianual) | ~**3%** |

Los clientes con contrato **mes a mes evaden 14 veces más** que aquellos con contrato bianual. Esto sugiere que la falta de compromiso a largo plazo es un factor crítico de riesgo.

---

### 2.3 Hallazgo 2 – Los Clientes Nuevos son los Más Vulnerables

El análisis de antigüedad revela una relación clara:
- **Clientes que evaden:** antigüedad media de ~**18 meses**
- **Clientes que permanecen:** antigüedad media de ~**38 meses**

La distribución KDE muestra un **pico de evasión en los primeros 1-12 meses**. Los clientes que superan los 24 meses de relación con la empresa tienen una tasa de churn marcadamente menor. Esto indica que el período crítico de retención es el **primer año de vida del cliente**.

---

### 2.4 Hallazgo 3 – Mayor Gasto Mensual se Asocia con Mayor Evasión

Contraintuitivamente, los clientes que pagan **más** por mes son los que más se van:
- **Clientes que evaden:** cargo mensual medio de ~**$74 USD**
- **Clientes que permanecen:** cargo mensual medio de ~**$61 USD**

Esto refleja un problema de **percepción de valor**: los clientes que contratan más servicios pero con contrato mensual sienten que el precio no justifica la permanencia. El boxplot confirmó que el segmento de evasión tiene una mediana de cargos significativamente más alta.

---

### 2.5 Hallazgo 4 – La Antigüedad Correlaciona Negativamente con la Evasión

La matriz de correlación (heatmap) confirmó que la **Antigüedad tiene la correlación negativa más fuerte con la Evasión** (r ≈ -0.35), siendo el predictor numérico más relevante. Los `Cargos_Totales` también tienen una correlación negativa fuerte porque un cargo total alto implica mayor antigüedad acumulada.

---

### 2.6 Hallazgo 5 – Los Clientes con Pocos Servicios Evaden Más

El análisis de `Total_Servicios` mostró que los clientes con **1 o 2 servicios contratados** tienen tasas de evasión más altas, mientras que quienes contratan **5 o más servicios** presentan mayor fidelidad. El "efecto ancla" de múltiples servicios integrados aumenta el costo de cambio (*switching cost*).

## 3. Recomendaciones Estratégicas

Basadas en los hallazgos del análisis, se proponen tres recomendaciones prioritarias para reducir el Churn:

---

### 🎯 Recomendación 1 – Programa de Migración a Contratos Anuales

**Problema:** El 43% de los clientes con contrato mes a mes evaden. Este segmento representa el mayor volumen de pérdida.  
**Acción:** Diseñar una campaña de incentivos **al momento del alta** y en los primeros 6 meses para migrar contratos mensuales a anuales:
- Descuentos del 10-15% en el primer año al firmar contrato anual
- Beneficios exclusivos (servicios adicionales gratuitos por 3 meses)
- Comunicación de ahorro anual esperado en formato numérico concreto ($X de ahorro)

**Impacto esperado:** Reducir la tasa de evasión del segmento mensual del 43% al 15-20%, lo que equivaldría a retener ~800-1,000 clientes adicionales al año.

---

### 🎯 Recomendación 2 – Programa de Onboarding Intensivo (Primeros 12 Meses)

**Problema:** El pico de evasión ocurre durante el primer año de vida del cliente. La empresa pierde clientes antes de que se genere la lealtad de largo plazo.  
**Acción:** Implementar un **programa de onboarding estructurado** para nuevos clientes (0-12 meses):
- Check-ins proactivos en los meses 1, 3 y 6 (por agente o chatbot)
- Encuesta de satisfacción (NPS) al mes 3 con seguimiento personalizado
- Incentivo de fidelidad en el mes 11 (antes de la renovación) para clientes con contrato anual
- Identificar señales de riesgo temprano: clientes que reducen uso del servicio o tienen quejas recurrentes

**Impacto esperado:** Reducir la evasión en el primer año de ~40% a ~25%, mejorando significativamente el LTV (Lifetime Value) promedio del cliente.

---

### 🎯 Recomendación 3 – Revisión de Propuesta de Valor para Clientes de Alto Cargo

**Problema:** Los clientes que pagan más de $65/mes evaden más que los que pagan menos, lo que indica una brecha de percepción de valor.  
**Acción:**  
- Realizar entrevistas de salida (*exit interviews*) para entender las razones de cancelación en el segmento >$70/mes
- Revisar el **precio de los paquetes premium** y comparar contra la competencia local
- Crear paquetes de "bundle" atractivos que integren 4+ servicios con precio competitivo (Internet + TV + Seguridad + Soporte)
- Comunicar el **valor diferencial** (uptime, velocidad, soporte) de forma más efectiva a este segmento

**Impacto esperado:** Mejorar la retención en el segmento premium en 15-20 puntos porcentuales, que al ser el segmento de mayor ARPU, tendría el mayor impacto en ingresos recurrentes.

---

## 4. Conclusión

El análisis confirma que la evasión de clientes en TelecomX no es aleatoria: está **concentrada en segmentos bien definidos** (contratos mensuales, clientes nuevos, alto costo mensual) que pueden ser abordados con estrategias específicas. La implementación de las tres recomendaciones propuestas, en orden de prioridad, tiene el potencial de reducir la tasa de churn global del 26% al rango del **15-18%** en un horizonte de 12-18 meses, con un impacto directo y medible en los ingresos.

---
*Análisis desarrollado como parte de la formación en Data Science de [Alura Latam](https://www.aluracursos.com/) – One Oracle Next Education (ONE).*